**Fig. 4A and Video S1: whole-tissue run on the adult trunk epidermis strip with the DSM pair**

Needs a GPU.

In [ ]:
# Working directory: data/fig4/ under the repository root, where the Fig. 4 deposit folder is downloaded
# (domain-specialized_finetuned_bootstrap/ = the DSM model pair, one folder per sample with its
# input volume and the final mask, wound_finetuning/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig4")
print("working directory:", os.getcwd())


# EpiCycle ScaledSegmentation Pipeline

Run 3D cell segmentation, reconstruction, and assembly on tiled image data.

**Usage:** Fill in the paths and parameters below, then Run All.

## 1. Paths & Input

In [ ]:
# ===== PATHS (edit these) =====

# Input Z-stack TIFF (the original microscope file name is in the README) — set this to tile from scratch, or leave None if already tiled
INPUT_TIF = "adult_skin_strip/input.tif"   # Set to None if tiles already exist

# Working directory (tiles + outputs go here)
# If INPUT_TIF is set, tiles will be created at WORK_DIR/tiles/
# If INPUT_TIF is None, WORK_DIR should contain existing tile subdirectories
WORK_DIR = "adult_skin_strip"

# CellPose model paths (.pth files)
LATERAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True"
AXIAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True"

# Output directory name (created inside WORK_DIR)
OUTPUT_NAME = "WT_AdultSkin_Strip"

## 2. Resource Configuration

In [ ]:
# ===== RESOURCES (auto-detected from the job's allocation if None) =====

N_GPUS = None            # None = auto-detect from SLURM/CUDA
N_CPU_PER_WORKER = None  # None = auto (total_cores / n_gpus in parallel mode)

## 3. Stage Selection & Tiling Parameters

In [ ]:
# Which stages to run (set False to skip)
RUN_TILING = INPUT_TIF is not None   # Auto: tile if input TIFF provided
# RUN_TILING = False
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_ASSEMBLY = True 
RUN_GLOBAL_RECONSTRUCTION = True

# ===== TILING (Stage 0) =====
PATCH_SIZE = 1280           # Target XY patch size in pixels
# OVERLAP is shared with assembly (set in Section 6)
OVERLAP = 0.5

Z_THRESHOLD_METHOD = "hard"  # "hard", "percentile", or "default"
Z_THRESHOLD_VALUE = 0.07     # Intensity threshold for Z-range detection
CHANNEL_INDEX = 1            # Which channel for Z-range analysis
AUTO_CORRECT_PATCH = False   # Auto-find valid patch size if exact tiling fails

## 4. Segmentation Parameters

In [ ]:
# ===== SEGMENTATION (Stage 1) =====
# Recorded for reference: the stage is built without segment_params below, so the engine
# defaults were used (channels [2, 1], cell probability 0, flow threshold 0.2). The flow
# threshold acts on the 2D masks only; the 3D reconstruction uses the probability and flow maps.

# GPU allocation strategy: "auto", "fixed" (3-GPU triplets), "round_robin"
GPU_STRATEGY = "auto"

# CellPose parameters
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
CHANNELS = [2, 1]      # [signal, nucleus] — adjust for your data
BATCH_SIZE = 1024

## 5. Reconstruction Parameters

In [ ]:
# ===== RECONSTRUCTION (Stage 2) =====

# Per-tile direct reconstruction: the engine's u-Segment3D defaults, with gradient renormalisation
# off as in the engine version the runs used (EpiCycle v0.1.2 defaults to on).
RECON_RENORM_GRADIENT = False

# "sequential" (one tile at a time, all cores) or "parallel" (multiple tiles)
RECON_MODE = "parallel"
# RECON_MODE = "sequential"

# Max concurrent tiles in parallel mode (None = n_gpus)
# Rule of thumb: 1900 TB node / ~450 GB per 1920x1920 tile ≈ 4 max
RECON_MAX_WORKERS = None

# Post-processing
SAVE_PROB3D = True
SAVE_FLOW3D = True
SAVE_BINARY = True
BINARY_ERODE = True
BINARY_KERNEL_SIZE = 3
BINARY_BORDER_MARGIN = 2
BINARY_FILL_HOLES = True

# Delete stage1 intermediates after reconstruction to save disk
CLEANUP_INTERMEDIATES = False

## 6. Assembly Parameters

In [ ]:
# ===== ASSEMBLY (Stage 3) =====

# Full volume dimensions in pixels (H, W)
FULL_SIZE = (1280, 7680)

# Tile size and overlap (must match how tiles were cut)
TILE_SIZE = PATCH_SIZE

# Number of Z slices (None = auto-detect from first tile)
DEPTH = None

# Which volumes to stitch
STITCH_MASK = True
STITCH_PROB = True
STITCH_BINARY = True

# Overlap blending for probabilities: "average" or "max"
PROB_STITCH_MODE = "max"

## 7. Global Indirect Reconstruction Parameters

In [ ]:
# ===== GLOBAL INDIRECT RECONSTRUCTION (Stage 4) =====

DOWNSAMPLE_FACTOR = 2          # Downsample stitched volume by this factor
GLOBAL_MIN_SIZE = 1000         # Remove objects smaller than this (voxels, after downsampling)
USE_PRECOMPUTED_BINARY = True  # Use stitched binary mask as foreground constraint

# Gradient descent parameters
# GLOBAL_N_ITER = 30            # Gradient descent iterations (more = finer, slower)
GLOBAL_N_ITER = 300            # Gradient descent iterations (more = finer, slower)
GLOBAL_MOMENTA = 0.9          # Momentum for gradient descent
# GLOBAL_GRADIENT_DECAY = 0.001    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.01    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.1    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.05    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.25    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
GLOBAL_GRADIENT_DECAY = 0.3    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower

GLOBAL_RENORM_GRADIENT = True  # Renormalize gradient at each step

# Indirect method parameters
# GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_SMOOTH_SKEL_SIGMA = 1   # Skeleton smoothing sigma
GLOBAL_N_CPU = 64              # CPU threads for multiprocess watershed
GLOBAL_TILE_SHAPE = None       # Tile shape for watershed (None = auto)

## 8. Setup & Pre-flight Check

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# Resolve GPU count
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus
n_gpus = N_GPUS or get_allocated_gpus()

work_dir = Path(WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

# Output lives inside the working directory
output_dir = work_dir / OUTPUT_NAME
output_dir.mkdir(exist_ok=True)
log_dir = output_dir / "logs"
log_dir.mkdir(exist_ok=True)

# Tiles directory
tiles_dir = work_dir / "tiles"

if RUN_TILING:
    print(f"Input TIFF: {INPUT_TIF}")
    print(f"Tiles will be created in: {tiles_dir}")
    print(f"Patch size: {PATCH_SIZE}, overlap: {OVERLAP}")
else:
    # Discover existing tile directories
    tile_dirs = sorted([
        str(p) for p in work_dir.iterdir()
        if p.is_dir() and not p.name.startswith(("stage1", "stage2", "stage3", "stage4", "logs", ".", "epicycle"))
        and p.name != OUTPUT_NAME
    ])
    print(f"Found {len(tile_dirs)} existing tile directories in {work_dir}")
    for td in tile_dirs[:5]:
        print(f"  {Path(td).name}")
    if len(tile_dirs) > 5:
        print(f"  ... and {len(tile_dirs) - 5} more")

print(f"\nGPUs: {n_gpus}")
print(f"Output: {output_dir}")

Found 1 existing tile directories in <run>/WT_Adult_Skin/WT_AdultSkin_Strip
  tiles

GPUs: 6
Output: <run>/WT_Adult_Skin/WT_AdultSkin_Strip/WT_AdultSkin_Strip


## 9. Build & Run Pipeline

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline

pipeline = EpiCyclePipeline(
    work_dir=str(work_dir),
    n_gpus=n_gpus,
    log_dir=str(log_dir),
)

# --- Stage 0: Tiling ---
if RUN_TILING:
    from packages.epicycle_scaledseg.core.tiling import TilingStage
    pipeline.add_stage(TilingStage(
        input_tif=INPUT_TIF,
        output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD,
        threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX,
        auto_correct=AUTO_CORRECT_PATCH,
    ))
    # Tiling creates tile_dirs; use placeholder for now
    tile_dirs = [str(work_dir)]

# --- Stage 1: Segmentation ---
if RUN_SEGMENTATION:
    from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
    pipeline.add_stage(SegmentationStage(
        lateral_model=LATERAL_MODEL,
        axial_model=AXIAL_MODEL,
        out_root=str(output_dir / "stage1_segmentation"),
        gpu_strategy=GPU_STRATEGY,
    ))

# --- Stage 2: Reconstruction ---
if RUN_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_reconstruct_params
    reconstruct_params = get_default_reconstruct_params()
    reconstruct_params['gradient_descent']['renorm_gradient'] = RECON_RENORM_GRADIENT
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE,
        reconstruct_params=reconstruct_params,
        max_workers=RECON_MAX_WORKERS,
        n_cpu_per_worker=N_CPU_PER_WORKER,
        save_prob3d=SAVE_PROB3D,
        save_flow3d=SAVE_FLOW3D,
        save_binary=SAVE_BINARY,
        binary_erode=BINARY_ERODE,
        binary_kernel_size=BINARY_KERNEL_SIZE,
        binary_border_margin=BINARY_BORDER_MARGIN,
        binary_fill_holes=BINARY_FILL_HOLES,
        cleanup_intermediates=CLEANUP_INTERMEDIATES,
    ))

# --- Stage 3: Assembly ---
if RUN_ASSEMBLY:
    from packages.epicycle_scaledseg.core.assembly import AssemblyStage
    pipeline.add_stage(AssemblyStage(
        recon_root=str(output_dir / "stage2_reconstruction"),
        output_dir=str(output_dir / "stage3_assembly"),
        full_size=FULL_SIZE,
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        depth=DEPTH,
        stitch_mask=STITCH_MASK,
        stitch_prob=STITCH_PROB and SAVE_PROB3D,
        stitch_binary=STITCH_BINARY and SAVE_BINARY,
        prob_stitch_mode=PROB_STITCH_MODE,
    ))

# --- Stage 4: Global Indirect Reconstruction ---
if RUN_GLOBAL_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.global_reconstruction import GlobalReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_indirect_reconstruct_params

    indirect_params = get_default_indirect_reconstruct_params(n_cpu=GLOBAL_N_CPU)
    indirect_params['gradient_descent']['n_iter'] = GLOBAL_N_ITER
    indirect_params['gradient_descent']['momenta'] = GLOBAL_MOMENTA
    indirect_params['gradient_descent']['gradient_decay'] = GLOBAL_GRADIENT_DECAY
    indirect_params['gradient_descent']['renorm_gradient'] = GLOBAL_RENORM_GRADIENT
    indirect_params['indirect_method']['dtform_method'] = GLOBAL_DTFORM_METHOD
    indirect_params['indirect_method']['smooth_skel_sigma'] = GLOBAL_SMOOTH_SKEL_SIGMA
    if GLOBAL_TILE_SHAPE is not None:
        indirect_params['gradient_descent']['tile_shape'] = GLOBAL_TILE_SHAPE

    pipeline.add_stage(GlobalReconstructionStage(
        assembly_dir=str(output_dir / "stage3_assembly"),
        output_dir=str(output_dir / "stage4_global"),
        downsample_factor=DOWNSAMPLE_FACTOR,
        min_size=GLOBAL_MIN_SIZE,
        use_precomputed_binary=USE_PRECOMPUTED_BINARY,
        indirect_params=indirect_params,
        n_cpu=GLOBAL_N_CPU,
        tile_shape=GLOBAL_TILE_SHAPE,
    ))

print(f"Pipeline: {len(pipeline.stages)} stages")
for s in pipeline.stages:
    print(f"  - {s.name}")
print(f"Tile dirs: {len(tile_dirs)}")
print(f"Output: {output_dir}")

# ===== RUN =====
result = pipeline.run(tile_dirs)
print()
print(result.summary())

2026-02-04 20:45:15,547 epicycle.pipeline INFO Pipeline starting: 1 stage(s), 1 tile(s), 6 GPU(s)
2026-02-04 20:45:15,547 epicycle.pipeline INFO Run parameters saved to data/WT_Adult_Skin/WT_AdultSkin_Strip/WT_AdultSkin_Strip/logs/run_params.json
2026-02-04 20:45:15,547 epicycle.pipeline INFO === Stage: global_reconstruction ===
2026-02-04 20:45:15,548 epicycle.global_reconstruction INFO Stage 'global_reconstruction' starting with 1 tile(s)
2026-02-04 20:45:15,548 epicycle.global_reconstruction INFO Parameters: {"downsample_factor": 2, "min_size": 1000, "use_precomputed_binary": true, "indirect_params": {"combine_cell_gradients": {"ksize": 1, "alpha": 0.5, "eps": 1e-20, "smooth_sigma": 1, "post_sigma": 0}, "combine_cell_probs": {"ksize": 1, "alpha": 0.5, "eps": 1e-20, "cellpose_prob_mask": true, "smooth_sigma": 0, "threshold_level": 1, "threshold_n_levels": 3, "apply_one_d_p_thresh": true, "prob_thresh": null, "min_prob_thresh": 0.0}, "postprocess_binary": {"binary_closing": 1, "remove

Pipeline: 1 stages
  - global_reconstruction
Tile dirs: 1
Output: <run>/WT_Adult_Skin/WT_AdultSkin_Strip/WT_AdultSkin_Strip


2026-02-04 20:45:15,777 epicycle.core.global_reconstruction INFO Loading stitched mask from <run>/WT_Adult_Skin/WT_AdultSkin_Strip/WT_AdultSkin_Strip/stage3_assembly/full_mask.tif
2026-02-04 20:45:33,640 epicycle.core.global_reconstruction INFO   mask shape: (1886, 1280, 7680), dtype: uint32
2026-02-04 20:45:33,641 epicycle.core.global_reconstruction INFO Downsampling mask by 2x (nearest-neighbor)
2026-02-04 20:45:36,893 epicycle.core.global_reconstruction INFO   downsampled mask shape: (943, 640, 3840)
2026-02-04 20:45:36,893 epicycle.core.global_reconstruction INFO Loading stitched binary from <run>/WT_Adult_Skin/WT_AdultSkin_Strip/WT_AdultSkin_Strip/stage3_assembly/full_binary.npy
2026-02-04 20:45:41,153 epicycle.core.global_reconstruction INFO Downsampling binary by 2x (max-pool)
2026-02-04 20:47:55,656 epicycle.core.global_reconstruction INFO   downsampled binary shape: (943, 640, 3840)
2026-02-04 20:47:55,657 epicycle.core.global_reconstruction INFO Cleaning mask (min_size=1000)


[1039  832 4032] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  924
start dynamics
Tile id 188 0 0
Tile id 104 0 0
Tile id 152 0 0
Tile id 20 0 0
Tile id 72 0 0
Tile id 32 0 0
Tile idTile id 40 0 0
 156 0 0
Tile id 180 0 0
Tile id 56 0 0
Tile id 140 0 0
Tile idTile id 96 0 0
 48 0 0
Tile id 204 0 0
Tile id 60 0 0
Tile id 208 0 0
Tile id 144 0 0
Tile id 124 0 0
Tile id 220 0 0
Tile id 12 0 0
Tile id 200 0 0
Tile id 68 0 0
Tile id 92 0 0
Tile id 244 0 0
Tile id 160 0 0
Tile id 80 0 0
Tile id 16 0 0
Tile id 100 0 0
Tile id 36 0 0
Tile id 64 0 0
Tile id 8 0 0
Tile id 136 0 0
Tile id 132 0 0
Tile id 112 0 0
Tile id 164 0 0
Tile id 120 0 0
Tile id 52 0 0
Tile id 224 0 0
Tile id 44 0 0
Tile id 28 0 0
Tile id 248 0 0
Tile id 24 0 0
Tile id 228 0 0
Tile id 76 0 0
Tile id 184 0 0
Tile id 116 0 0
Tile id 240 0 0
Tile id 148 0 0
Tile id 4 138 138
Tile id 209 0 0
Tile id 153 0 0
Tile id 41 0 0
Tile id 125 0 0
Tile id 161 0 0
Tile id 21 0 0
Tile id 33 0 0
Tile id 141 0 0
Tile id 157 0

2026-02-04 21:33:11,057 epicycle.core.global_reconstruction INFO Global reconstruction complete: shape=(943, 640, 3840), n_labels=9379, elapsed=2786.4s
2026-02-04 21:33:11,059 epicycle.core.global_reconstruction INFO Saving outputs to <run>/WT_Adult_Skin/WT_AdultSkin_Strip/WT_AdultSkin_Strip/stage4_global
2026-02-04 21:36:53,701 epicycle.pipeline INFO Stage 'global_reconstruction': 1/1 succeeded (3098.2s)
2026-02-04 21:36:53,702 epicycle.global_reconstruction INFO Stage 'global_reconstruction' finished: 1/1 succeeded (3098.2s)
2026-02-04 21:36:53,703 epicycle.pipeline WARNING No tiles remaining — stopping pipeline.
2026-02-04 21:36:53,704 epicycle.pipeline INFO Pipeline: 1/1 tiles (3098.2s total)
  global_reconstruction: 1/1 succeeded (3098.2s)



Pipeline: 1/1 tiles (3098.2s total)
  global_reconstruction: 1/1 succeeded (3098.2s)


## 10. Check Status

In [ ]:
# Quick status check — how many tiles completed each stage?
seg_root = output_dir / "stage1_segmentation"
recon_root = output_dir / "stage2_reconstruction"

seg_done = sum(1 for td in tile_dirs if (seg_root / Path(td).name / "SEGMENTATION_DONE").exists())
recon_done = sum(1 for td in tile_dirs if (recon_root / Path(td).name / f"{Path(td).name}_mask.tif").exists())

print(f"Stage 1 (Segmentation):   {seg_done}/{len(tile_dirs)} complete")
print(f"Stage 2 (Reconstruction): {recon_done}/{len(tile_dirs)} complete")
print(f"Stage 3 (Assembly):       check {output_dir / 'stage3_assembly'}")
print(f"Stage 4 (Global Recon):   {'done' if (output_dir / 'stage4_global' / 'global_segmentation.tif').exists() else 'not done'}")

Stage 1 (Segmentation):   0/1 complete
Stage 2 (Reconstruction): 0/1 complete
Stage 3 (Assembly):       check <run>/WT_Adult_Skin/WT_AdultSkin_Strip/WT_AdultSkin_Strip/stage3_assembly
Stage 4 (Global Recon):   done
